# LLM Request SDK Quickstart

서버화된 `llm-request`를 SDK로 사용하는 가이드.

**구조:**
```
이 노트북 (SDK)                                   llm-request Service (8101)
┌──────────────┐                                 ┌──────────────────────────────┐
│ llm_request  │──HTTP (files=[FileInput,...]) ─→│ ingest_files → file-manager   │
│ _async(...)  │                                 │ MDB 캐시 조회                  │
│              │                                 │ 파일 처리 (inline/Files API)  │
│              │                                 │ LLM 호출 (Gemini)              │
│              │←── {result, meta} ─────────────│ 결과 저장 + 반환               │
└──────────────┘                                 └──────────────────────────────┘
```

에이전트는 **file-manager SDK를 직접 쓰지 않는다** — 로컬 파일에서 `FileInput`(meta + bytes)만
만들어 `llm_request_async(files=[...])`에 넘기면 서비스가 내부에서 file-manager 등록 및 처리.

**주요 변경점 (라이브러리 → SDK+Service):**

| | 라이브러리 (v1) | SDK (현재) |
|--|---------------|------------|
| 설정 | URL 3개 + configure 3번 | `LLM_REQUEST_URL` 하나 |
| 파일 전달 | `file_paths=["report.pdf"]` | `files=[FileInput(...)]` (meta + bytes) |
| 시간 설정 | `ttl_seconds=`, `context_cache_ttl=` 각각 | `timings={"mdb_ttl": ..., "context_cache_ttl": ...}` |

## 설정

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()

os.environ["LLM_REQUEST_URL"]  = "http://localhost:8101"
TEST_DB = "kw_ai_dev"

print(f"LLM_REQUEST_URL  = {os.environ['LLM_REQUEST_URL']}")
print(f"TEST_DB          = {TEST_DB}")

LLM_REQUEST_URL  = http://localhost:8101
TEST_DB          = kw_ai_dev


## 타이밍 유틸리티

각 단계별 소요 시간과 네트워크 흐름을 추적하는 헬퍼.

In [2]:
import time
import mimetypes
from contextlib import contextmanager
from pathlib import Path

from llm_request import FileInput

@contextmanager
def timer(label: str):
    """구간별 소요 시간 측정."""
    start = time.perf_counter()
    print(f"  [{label}] 시작...")
    yield
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  [{label}] 완료: {elapsed:.0f}ms")

def print_flow(title: str, meta: dict, elapsed_ms: float):
    """요청 흐름 요약 출력."""
    source = meta.get("source", "?")
    cache_hit = meta.get("cache_hit", False)
    cache_key = meta.get("cache_key", "")[:16]

    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(f"  소스:      {source} ({'MDB 캐시 히트' if cache_hit else 'LLM 호출'})")
    print(f"  캐시 키:   {cache_key}...")
    print(f"  총 소요:   {elapsed_ms:.0f}ms")

    if meta.get("route_reason"):
        print(f"  경로:      {meta['route_reason']}")
    print()

def build_file_input(path) -> FileInput:
    """로컬 파일 → FileInput (meta 5필드 + bytes).

    file-manager 호출은 llm-request 서비스가 내부에서 처리하므로
    에이전트는 로컬 파일로부터 FileInput만 만들면 된다.
    """
    p = Path(path)
    return FileInput(
        file_name=p.name,
        file_size=p.stat().st_size,
        file_mtime=int(p.stat().st_mtime),
        mime_type=mimetypes.guess_type(p.name)[0] or "application/octet-stream",
        extension=p.suffix.lstrip("."),
        data=p.read_bytes(),
    )

ModuleNotFoundError: No module named 'llm_request'

---
## 1. 기본 호출 — 텍스트만 (파일 없음)

```
SDK                              Service (8101)
 │  POST /v1/request             │
 │  {llm_input, cache_mode}  ──→ │ → MDB 캐시 조회
 │                               │ → MISS → LLM 호출 (Gemini)
 │  ←── {result, meta}          │ → MDB 캐시 저장
```

In [ ]:
from llm_request import llm_request_async

# --- 1회차: LLM 직접 호출 (캐시 miss) ---
start = time.perf_counter()
result, meta = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("1회차: 첫 호출 (캐시 miss → LLM)", meta, elapsed)
print(f"  응답: {result.get('text', '')[:100]}")

In [ ]:
# --- 2회차: 동일 입력 → MDB 캐시 히트 (LLM 호출 없음) ---
start = time.perf_counter()
result2, meta2 = await llm_request_async(
    {
        "contents": "1+1은?",
        "model": "gemini-2.5-flash",
    },
    database="kw_ai_dev",
)
elapsed2 = (time.perf_counter() - start) * 1000

print_flow("2회차: 동일 입력 (MDB 캐시 히트)", meta2, elapsed2)
print(f"  응답: {result2.get('text', '')[:100]}")
print(f"  속도 비교: {elapsed:.0f}ms → {elapsed2:.0f}ms ({elapsed/max(elapsed2,0.1):.1f}x 빠름)")

---
## 2. 파일 포함 호출

```
SDK (로컬)                                       llm-request Service (8101)
 │                                                │
 │  1. build_file_input (로컬 bytes + 메타)        │
 │                                                │
 │  2. POST /v1/request (files=[FileInput])  ───→ │ ingest_files
 │                                                │   └ file-manager check/upload
 │                                                │ MDB 캐시 조회 → miss
 │                                                │ LLM 호출 (Gemini)
 │  ←── {result, meta} ───────────────────────── │ MDB 캐시 저장
```

In [ ]:
from llm_request import llm_request_async

test_file = "small_file.pdf"
if not Path(test_file).exists():
    print(f"⚠️ {test_file}이 없습니다. 아래 셀은 건너뛰세요.")
else:
    print(f"파일: {test_file} ({Path(test_file).stat().st_size / 1024:.0f}KB)")
    print()

    # 1. 로컬 파일 → FileInput 구성
    with timer("build_file_input"):
        fi = build_file_input(test_file)
    print(f"    → mime={fi.mime_type}, size={fi.file_size}")
    print()

    # 2. llm_request_async (files= 전달)
    start = time.perf_counter()
    with timer("llm_request_async 전체"):
        result, meta = await llm_request_async(
            {
                "contents": "이 문서를 한 문장으로 요약해줘",
                "model": "gemini-2.5-flash",
            },
            files=[fi],
            database=TEST_DB,
        )
    elapsed = (time.perf_counter() - start) * 1000

    print()
    print_flow("파일 포함 호출 (1회차)", meta, elapsed)
    print(f"  응답: {result.get('text', '')[:200]}")

---
## 3. done_check + fetch_by_cache_key

캐시 존재 여부만 확인하고, 필요 시 직접 조회.

```
SDK                              Service (8101)
 │  POST /v1/done-check          │
 │  {llm_input}             ──→  │ → MDB 캐시 조회
 │  ←── {cache_key, cached}     │
 │                               │
 │  POST /v1/fetch               │
 │  {cache_key}             ──→  │ → MDB에서 결과 반환
 │  ←── {result, meta}          │
```

In [ ]:
from llm_request import done_check, fetch_by_cache_key

# --- done_check: 캐시 존재 여부 확인 ---
with timer("done_check"):
    dc = await done_check(
        {"contents": "1+1은?", "model": "gemini-2.5-flash"},
        database="kw_ai_dev",
    )

print(f"  cache_key: {dc['cache_key'][:16]}...")
print(f"  cache_hit: {dc['cache_hit']}")
print()

# --- fetch_by_cache_key: 캐시 키로 직접 조회 ---
if dc["cache_hit"]:
    with timer("fetch_by_cache_key"):
        result, meta = await fetch_by_cache_key(
            dc["cache_key"],
            database="kw_ai_dev",
        )
    print(f"  응답: {result.get('text', '')[:100]}")
else:
    print("  캐시 없음 — 위의 1번 셀을 먼저 실행하세요")

---
## 4. 기존 라이브러리와 비교

| | 라이브러리 (v1) | SDK (현재) |
|--|---------------|------------|
| **설정** | MDB_HANDLER_URL + LLM_HANDLER_URL + REDIS_URL + configure 3개 | `LLM_REQUEST_URL` 하나 |
| **의존성** | llm-handler, mdb-handler, redis-handler 전부 | `llm_request` 패키지 하나 |
| **사용 코드** | `llm_request_async(...)` | `llm_request_async(...)` (동일) |
| **파일 전달** | `file_paths=["report.pdf"]` (SDK가 직접 Files API 호출) | `files=[FileInput]` (meta + bytes를 서비스로 전송) |
| **파일 등록** | SDK → Redis/Gemini 직접 처리 | 서비스 내부에서 file-manager 호출 |
| **캐시 조회** | SDK가 MDB 직접 조회 | llm-request Service가 조회 |
| **시간 설정** | `ttl_seconds=`, `context_cache_ttl=` 각각 | `timings={"mdb_ttl": ..., "context_cache_ttl": ...}` |

---
## 5. 시스템 프롬프트 + service_name

In [ ]:
start = time.perf_counter()
result, meta = await llm_request_async(
    {
        "contents": "ESG 평가 기준을 알려줘",
        "model": "gemini-2.5-flash",
        "system": "너는 ESG 전문가야. 한국어로 답변해.",
        "temperature": 0,
        "max_tokens": 200,
        "service_name": "esg_agent",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("시스템 프롬프트 + service_name", meta, elapsed)
print(f"  응답: {result.get('text', '')[:200]}")

---
## 6. 대용량 파일 (>50MB PDF → 자동 split + Files API)

```
SDK (로컬)                                       llm-request Service (8101)
 │                                                │
 │  1. build_file_input (로컬 bytes + 메타)        │
 │                                                │
 │  2. POST /v1/request (files=[FileInput])  ───→ │ ingest_files → file-manager 등록
 │                                                │ MDB 캐시 조회 → miss
 │                                                │ classify_files → Files API 경로
 │                                                │ split_pdf (50MB 초과 분할)
 │                                                │ Files API 업로드
 │                                                │ LLM 호출
 │  ←── {result, meta} ───────────────────────── │ MDB 캐시 저장
```

In [ ]:
from pathlib import Path

LARGE_PDF = "large_file.pdf"

if not Path(LARGE_PDF).exists():
    print(f"⚠️ 대용량 PDF를 찾을 수 없습니다: {LARGE_PDF}")
    print("   경로를 수정하거나 이 셀을 건너뛰세요.")
else:
    size_mb = Path(LARGE_PDF).stat().st_size / (1024 * 1024)
    print(f"파일: {Path(LARGE_PDF).name} ({size_mb:.1f}MB)")
    print()

    # 1. 로컬 파일 → FileInput 구성
    with timer("build_file_input"):
        fi_large = build_file_input(LARGE_PDF)
    print(f"    → mime={fi_large.mime_type}")
    print()

    # 2. 첫 호출 (서비스 내부에서 ingest + split + upload + LLM)
    start = time.perf_counter()
    with timer("대용량 PDF 전체"):
        result, meta = await llm_request_async(
            {
                "contents": "이 PDF의 제목을 한글 한 줄로.",
                "model": "gemini-2.5-flash",
                "max_tokens": 100,
            },
            files=[fi_large],
            database=TEST_DB,
        )
    elapsed = (time.perf_counter() - start) * 1000

    print_flow("대용량 PDF (1회차)", meta, elapsed)
    print(f"  응답: {result.get('text', '')[:200]}")

In [ ]:
# 6-2. 같은 파일 재호출 → MDB 캐시 히트 (업로드/LLM 호출 없음)
if Path(LARGE_PDF).exists():
    start = time.perf_counter()
    result2, meta2 = await llm_request_async(
        {
            "contents": "이 PDF의 제목을 한글 한 줄로.",
            "model": "gemini-2.5-flash",
            "max_tokens": 100,
        },
        files=[fi_large],
        database=TEST_DB,
    )
    elapsed2 = (time.perf_counter() - start) * 1000

    print_flow("대용량 PDF (2회차 — MDB 캐시 히트)", meta2, elapsed2)
    print(f"  응답: {result2.get('text', '')[:200]}")
    print(f"  속도 비교: {elapsed:.0f}ms → {elapsed2:.0f}ms")

---
## 7. 반환값 구조

모든 `llm_request_async` 함수는 `(result, meta)` 튜플을 반환합니다.

```python
result = {
    "text": "LLM 응답 텍스트",
    "model": "gemini-2.5-flash",
    "usage": {"input_tokens": 10, "output_tokens": 5, "total_tokens": 15},
    "finish_reason": "STOP",
    "request_id": "req-xxx",
    "function_calls": None,
}

meta = {
    "source": "live" | "cache",      # LLM 직접 호출 vs MDB 캐시
    "cache_hit": True | False,
    "cache_key": "sha256 hex (64자)",
    "metadata": {"service_name": "...", "model": "..."},
    "collection": "agent_llm",
}
```

### 캐시 키 동작 원리

`llm_input`에서 15개 필드 전부 추출(미입력/None → `""`) → 값 내부 정규화 → 키 정렬·JSON 직렬화 → SHA-256 해시

- **같은 입력 = 같은 캐시 키** → 캐시 히트
- **파일 첨부 시**: 파일 내용의 checksum이 캐시 키에 포함 → 같은 파일 + 같은 질문 = 캐시 히트 (파일명/URI 무관)

---
# Appendix: 에이전트 개발 패턴

`llm_request_async`를 사용하는 에이전트 개발 시 자주 쓰는 패턴입니다.

## 패턴 A: 단건 처리

단건 LLM 호출 + 캐시. 에이전트 내 단일 LLM 호출의 기본 패턴.

In [ ]:
from llm_request import llm_request_async

start = time.perf_counter()
result_a, meta_a = await llm_request_async(
    {
        "contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.",
        "model": "gemini-2.5-flash",
        "temperature": 0,
        "max_tokens": 150,
        "service_name": "fund_comment",
    },
    database="kw_ai_dev",
)
elapsed = (time.perf_counter() - start) * 1000

print_flow("패턴 A: 단건 처리", meta_a, elapsed)
print(f"  응답: {result_a.get('text', '')[:200]}")
if result_a.get("usage"):
    u = result_a["usage"]
    print(f"  토큰: {u.get('input_tokens', 0)} in / {u.get('output_tokens', 0)} out")

---
## 패턴 B: N건 병렬 — `llm_request_async` + 타이밍 로그

펀드마다 **DB 조회 지연**을 다르게 두어 LLM 요청 시각이 어긋나게 합니다.
로그에 `[경과초] 코드 DB완료` → `LLM시작` → `LLM끝(소요초)` 가 찍혀, **비동기 병렬 진행**을 확인할 수 있습니다.

```
005930: task → DB(즉시) → [LLM 2~4s] ─────────────┐
000660: task → DB(0.4s) → [LLM ...]              ├─ as_completed
035420: task → DB(0.8s) → [LLM ...]              │
068270: task → DB(1.2s) → [LLM ...]              ┘
```

In [ ]:
import asyncio
import time

from llm_request import llm_request_async

FUNDS = [
    {"code": "005930", "name": "삼성전자",  "sector": "반도체"},
    {"code": "000660", "name": "SK하이닉스", "sector": "반도체"},
    {"code": "035420", "name": "NAVER",      "sector": "IT"},
    {"code": "068270", "name": "셀트리온",   "sector": "바이오"},
]

DB_STAGGER_SEC = 0.42


async def process_fund(fund: dict, idx: int, wall0: float) -> dict:
    """DB 조회(시뮬) 후 LLM 호출."""
    # DB 조회 시뮬레이션 (stagger)
    await asyncio.sleep(DB_STAGGER_SEC * idx)
    t_db_done = time.perf_counter() - wall0
    print(f"  [{t_db_done:.2f}s] {fund['code']} DB 완료")

    # LLM 호출
    t_llm_start = time.perf_counter() - wall0
    result, meta = await llm_request_async(
        {
            "contents": (
                f"{fund['name']}({fund['code']})의 {fund['sector']} 사업 현황을 "
                f"투자 관점에서 간결한 한 문단(3~4문장)으로 설명해줘."
            ),
            "model": "gemini-2.5-flash",
            "temperature": 0.3,
            "max_tokens": 300,
            "service_name": "fund_comment",
        },
        database="kw_ai_dev",
    )
    t_llm_end = time.perf_counter() - wall0
    llm_sec = t_llm_end - t_llm_start

    print(f"  [{t_llm_end:.2f}s] {fund['code']} LLM 끝 ({llm_sec:.2f}s, {meta.get('source', '?')})")

    return {
        "code": fund["code"],
        "name": fund["name"],
        "text": result.get("text", ""),
        "source": meta.get("source", "?"),
        "t_db_done": t_db_done,
        "t_llm_start": t_llm_start,
        "t_llm_end": t_llm_end,
        "llm_sec": llm_sec,
    }


# 실행
wall0 = time.perf_counter()
tasks = [asyncio.create_task(process_fund(f, i, wall0)) for i, f in enumerate(FUNDS)]
many_results = []
for coro in asyncio.as_completed(tasks):
    many_results.append(await coro)

total = time.perf_counter() - wall0
print(f"\n  전체 소요: {total:.2f}s (순차 시 ~{sum(r['llm_sec'] for r in many_results):.2f}s)")

### 결과 확인 — 요약 표

`LLM시작` 시각이 DB끝 직후에 연달아 찍히면 비동기 병렬 진행 확인.

In [ ]:
rows = sorted(many_results, key=lambda x: x["code"])
hdr = f"{'코드':<8} {'DB끝(s)':>8} {'LLM시작(s)':>10} {'LLM끝(s)':>9} {'순수LLM(s)':>10} {'source':<8}"
print(hdr)
print("-" * len(hdr))
for r in rows:
    print(
        f"{r['code']:<8} "
        f"{r['t_db_done']:8.2f} "
        f"{r['t_llm_start']:10.2f} "
        f"{r['t_llm_end']:9.2f} "
        f"{r['llm_sec']:10.2f} "
        f"{r['source']:<8}"
    )

print()
for r in rows:
    preview = r["text"].replace("\n", " ")[:100]
    print(f"[{r['code']}] {preview}")

---
## 정리

테스트 데이터를 삭제합니다. (kw_ai_dev DB)

In [ ]:
# MDB 캐시 정리 — done_check으로 cache_key를 구해서 확인
from llm_request import done_check

test_inputs = [
    {"contents": "1+1은?", "model": "gemini-2.5-flash"},
    {"contents": "ESG 평가 기준을 알려줘", "model": "gemini-2.5-flash", "system": "너는 ESG 전문가야. 한국어로 답변해.", "temperature": 0, "max_tokens": 200, "service_name": "esg_agent"},
    {"contents": "삼성전자 반도체 사업 현황을 두 줄로 요약해줘.", "model": "gemini-2.5-flash", "temperature": 0, "max_tokens": 150, "service_name": "fund_comment"},
]

# 펀드 패턴 B 입력도 추가
for fund in FUNDS:
    test_inputs.append({
        "contents": (
            f"{fund['name']}({fund['code']})의 {fund['sector']} 사업 현황을 "
            f"투자 관점에서 간결한 한 문단(3~4문장)으로 설명해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.3,
        "max_tokens": 300,
        "service_name": "fund_comment",
    })

print("캐시 키 조회 중...")
for inp in test_inputs:
    dc = await done_check(inp, database="kw_ai_dev")
    status = "hit" if dc["cache_hit"] else "miss"
    print(f"  {dc['cache_key'][:16]}... ({status})")

print("\n정리 완료 (실제 삭제는 MDB에서 직접 수행)")